In [ ]:
!python -m pip install -q -U python-woc pandas requests tqdm
# please clear the output of this cell before checking it in

In [1]:
from woc.remote import WocMapsRemote
from tqdm import tqdm
from pathlib import Path
import pandas as pd
import requests
import json
import time

yournetid = "spate198"
woc = WocMapsRemote(base_url="https://worldofcode.org/api/")

# These are the 10 rows marked with my NetID in net2prj.csv.
projects = ['nansencenter_dapper', 'quantumblacklabs_kedro-viz', 'biod_sambamba', 'hungpham2511_toppra', 'bioimagesuiteweb_bisweb', 'carloscinelli_sensemakr', 'compphysvienna_n2p2', 'shraddhabarke_probe', 'openuc2_uc2-git', 'fangq_mcx']
project_urls = {'nansencenter_dapper': 'https://github.com/nansencenter/DAPPER', 'quantumblacklabs_kedro-viz': 'https://github.com/quantumblacklabs/kedro-viz', 'biod_sambamba': 'https://github.com/biod/sambamba', 'hungpham2511_toppra': 'https://github.com/hungpham2511/toppra', 'bioimagesuiteweb_bisweb': 'https://github.com/bioimagesuiteweb/bisweb', 'carloscinelli_sensemakr': 'https://github.com/carloscinelli/sensemakr', 'compphysvienna_n2p2': 'https://github.com/CompPhysVienna/n2p2', 'shraddhabarke_probe': 'https://github.com/shraddhabarke/probe', 'openuc2_uc2-git': 'https://github.com/openUC2/UC2-GIT', 'fangq_mcx': 'https://github.com/fangq/mcx'}

# Get each project's commit SHA-1 list. Save a cache so a stopped run can resume.
cache_dir = Path(yournetid + "_cache")
cache_dir.mkdir(exist_ok=True)
project_commits = {}
list_df_commits = []

for prj in projects:
    cache_file = cache_dir / f"p2c_{prj}.json"
    if cache_file.exists():
        commits = json.loads(cache_file.read_text())
    else:
        commits = woc.get_values("p2c", prj)
        cache_file.write_text(json.dumps(commits))
    project_commits[prj] = commits
    temp = pd.DataFrame(commits, columns=["sha1"])
    temp["project"] = prj
    list_df_commits.append(temp)

df = pd.concat(list_df_commits, ignore_index=True)
df.to_csv("df_commits.csv", index=False)
df.groupby("project").size()

nansencenter_dapper            2878
quantumblacklabs_kedro-viz    14652
biod_sambamba                  1351
hungpham2511_toppra            1444
bioimagesuiteweb_bisweb        3797
carloscinelli_sensemakr         498
compphysvienna_n2p2             771
shraddhabarke_probe             337
openuc2_uc2-git                 769
fangq_mcx                      2664

In [2]:
# The WoC batch endpoint accepts at most 10 commits per request.
# Results are appended to a cache so this long step can be resumed.
commit_cache = cache_dir / "commit_records.jsonl"
commit_data = {}
if commit_cache.exists():
    for line in commit_cache.read_text().splitlines():
        if line:
            record = json.loads(line)
            commit_data[record["commit_sha1"]] = record

all_sha1s = sorted(df["sha1"].unique())
missing = [sha for sha in all_sha1s if sha not in commit_data]
chunks = [missing[x:x+10] for x in range(0, len(missing), 10)]
api_errors = {}

with commit_cache.open("a") as cache:
    for chunk in tqdm(chunks):
        # Retry temporary rate-limit/server errors instead of losing the run.
        for attempt in range(7):
            try:
                res, err = woc.get_values_many("commit.tch", chunk)
                break
            except Exception:
                if attempt == 6:
                    raise
                time.sleep(min(60, 2 ** (attempt + 1)))

        api_errors.update(err)
        for sha, wrapped in res.items():
            commit = wrapped[0]
            record = {
                "commit_sha1": sha,
                "author": commit[2][0],
                "time": int(commit[2][1]),
                "commit message": commit[4],
            }
            commit_data[sha] = record
            cache.write(json.dumps(record, ensure_ascii=False) + "\n")
        cache.flush()
        time.sleep(0.5)

# A few p2c SHA-1 values may not exist in WoC's commit.tch map.
unresolved = [sha for sha in all_sha1s if sha not in commit_data]
(cache_dir / "unresolved_woc_sha1s.json").write_text(
    json.dumps({"commits": unresolved, "errors": api_errors}, indent=2)
)
print(f"Retrieved {len(commit_data):,} unique commit records")
print(f"WoC returned no commit object for {len(unresolved)} p2c SHA-1s")

Retrieved 29,145 unique commit records
WoC returned no commit object for 16 p2c SHA-1s


In [3]:
# What does one retrieved record look like?
first_sha = next(iter(commit_data))
print(commit_data[first_sha])

{'commit_sha1': '1eb3bb13a6923ed8873b86dcf3b323c7fd6d987f', 'author': 'Artem Tarasov <lomereiter@gmail.com>', 'time': 1335621640, 'commit message': 'decompressing bgzf\n'}


In [4]:
# Flatten the records into the exact columns requested by the assignment.
rows = []
for prj, commits in project_commits.items():
    for sha in commits:
        if sha not in commit_data:
            continue
        record = commit_data[sha]
        rows.append({
            "project_wocid": prj,
            "commit_sha1": sha,
            "author": record["author"],
            "time": record["time"],
            "commit message": record["commit message"],
        })

dfinf = pd.DataFrame(rows, columns=[
    "project_wocid", "commit_sha1", "author", "time", "commit message"
]).sort_values(["project_wocid", "time", "commit_sha1"], kind="stable")

In [5]:
# Check that the table has the right content.
dfinf.head(2)

project_wocid,commit_sha1,author,time,commit message
biod_sambamba,1eb3bb13a6923ed8873b86dcf3b323c7fd6d987f,Artem Tarasov <lomereiter@gmail.com>,1335621640,decompressing bgzf\n
biod_sambamba,d99d2fec9d833786b9e699a76027b375ed87c429,Artem Tarasov <lomereiter@gmail.com>,1335628908,fixes\n


In [6]:
# Save all 10 projects in one semicolon-separated file.
output_file = yournetid + "_project_summary.csv"
dfinf.to_csv(output_file, index=False, sep=";")
print(f"Saved {len(dfinf):,} rows to {output_file}")

Saved 29,145 rows to spate198_project_summary.csv


# Make sure you check in both the notebook and the CSV file to your fork.

In [7]:
# Get the requested live GitHub information for each repository.
github_rows = []
http = requests.Session()
http.headers.update({"Accept": "application/vnd.github+json",
                     "User-Agent": yournetid + "-cs445-miniproject2"})

for prj, url in project_urls.items():
    slug = url.removeprefix("https://github.com/").strip("/")
    repo_response = http.get(f"https://api.github.com/repos/{slug}", timeout=60)
    repo_response.raise_for_status()
    repo = repo_response.json()
    commit_response = http.get(
        f"https://api.github.com/repos/{slug}/commits",
        params={"sha": repo["default_branch"], "per_page": 1},
        timeout=60,
    )
    commit_response.raise_for_status()
    latest = commit_response.json()
    github_rows.append({
        "Project": prj,
        "nstars": repo["stargazers_count"],
        "nforks": repo["forks_count"],
        "lastGHCommitDate": latest[0]["commit"]["committer"]["date"] if latest else None,
    })

github_stats = pd.DataFrame(github_rows)
project_stats = (dfinf.groupby("project_wocid")
    .agg(ncommits=("commit_sha1", "size"),
         nauthors=("author", "nunique"),
         min_time=("time", "min"),
         max_time=("time", "max"))
    .reset_index()
    .rename(columns={"project_wocid": "Project"}))
project_stats["from"] = pd.to_datetime(
    project_stats.pop("min_time"), unit="s", utc=True
)
project_stats["to"] = pd.to_datetime(
    project_stats.pop("max_time"), unit="s", utc=True
)
project_stats = project_stats.merge(github_stats, on="Project", validate="one_to_one")
project_stats.to_csv(yournetid + "_project_stats.csv", index=False)
report = project_stats.rename(columns={
    "Project": "project",
    "ncommits": "number of commits",
    "nauthors": "number of authors",
    "from": "min time",
    "to": "max time",
    "nstars": "number of stars",
    "nforks": "number of forks",
    "lastGHCommitDate": "last commit date",
})[[
    "project", "number of stars", "number of forks", "last commit date",
    "number of commits", "number of authors", "max time", "min time"
]]
report

project,number of stars,number of forks,last commit date,number of commits,number of authors,max time,min time
nansencenter_dapper,415,132,2026-06-26T11:44:41Z,2873,26,2026-02-21T11:01:38+00:00,2016-07-03T17:02:21+00:00
quantumblacklabs_kedro-viz,761,126,2026-09-15T14:46:21Z,14652,139,2025-11-05T15:45:19+00:00,2018-04-16T16:05:44+00:00
biod_sambamba,613,103,2024-12-22T10:07:47Z,1351,41,2024-12-22T10:07:47+00:00,2012-04-28T14:00:40+00:00
hungpham2511_toppra,929,212,2026-08-28T05:00:11Z,1442,62,2025-10-03T13:33:57+00:00,2017-07-13T06:54:56+00:00
bioimagesuiteweb_bisweb,86,36,2026-09-10T14:03:45Z,3795,41,2025-03-26T16:39:18+00:00,2018-04-18T15:25:58+00:00
carloscinelli_sensemakr,100,17,2026-08-13T22:12:33Z,498,11,2024-11-11T03:07:42+00:00,2017-07-17T16:06:05+00:00
compphysvienna_n2p2,244,95,2025-03-17T11:03:50Z,771,39,2025-09-17T17:16:57+00:00,2018-07-25T12:29:17+00:00
shraddhabarke_probe,3,2,2024-09-11T13:58:56Z,337,5,2024-09-11T13:58:56+00:00,2019-11-26T23:17:28+00:00
openuc2_uc2-git,533,76,2026-08-09T17:25:59Z,769,21,2025-05-22T18:41:22+00:00,2018-08-18T12:02:25+00:00
fangq_mcx,193,96,2026-09-18T17:17:35Z,2657,41,2025-10-21T04:00:26+00:00,2009-02-25T02:37:09+00:00


# Requested project information

| project | number of stars | number of forks | last commit date | number of commits | number of authors | max time | min time |
| --- | --- | --- | --- | --- | --- | --- | --- |
| nansencenter_dapper | 415 | 132 | 2026-06-26T11:44:41Z | 2873 | 26 | 2026-02-21T11:01:38+00:00 | 2016-07-03T17:02:21+00:00 |
| quantumblacklabs_kedro-viz | 761 | 126 | 2026-09-15T14:46:21Z | 14652 | 139 | 2025-11-05T15:45:19+00:00 | 2018-04-16T16:05:44+00:00 |
| biod_sambamba | 613 | 103 | 2024-12-22T10:07:47Z | 1351 | 41 | 2024-12-22T10:07:47+00:00 | 2012-04-28T14:00:40+00:00 |
| hungpham2511_toppra | 929 | 212 | 2026-08-28T05:00:11Z | 1442 | 62 | 2025-10-03T13:33:57+00:00 | 2017-07-13T06:54:56+00:00 |
| bioimagesuiteweb_bisweb | 86 | 36 | 2026-09-10T14:03:45Z | 3795 | 41 | 2025-03-26T16:39:18+00:00 | 2018-04-18T15:25:58+00:00 |
| carloscinelli_sensemakr | 100 | 17 | 2026-08-13T22:12:33Z | 498 | 11 | 2024-11-11T03:07:42+00:00 | 2017-07-17T16:06:05+00:00 |
| compphysvienna_n2p2 | 244 | 95 | 2025-03-17T11:03:50Z | 771 | 39 | 2025-09-17T17:16:57+00:00 | 2018-07-25T12:29:17+00:00 |
| shraddhabarke_probe | 3 | 2 | 2024-09-11T13:58:56Z | 337 | 5 | 2024-09-11T13:58:56+00:00 | 2019-11-26T23:17:28+00:00 |
| openuc2_uc2-git | 533 | 76 | 2026-08-09T17:25:59Z | 769 | 21 | 2025-05-22T18:41:22+00:00 | 2018-08-18T12:02:25+00:00 |
| fangq_mcx | 193 | 96 | 2026-09-18T17:17:35Z | 2657 | 41 | 2025-10-21T04:00:26+00:00 | 2009-02-25T02:37:09+00:00 |

WoC's `p2c` map returned **16** SHA-1s that its own `commit.tch` map reported as missing. They were recorded in the cache and were not added as fake commit rows.

In [8]:
# Final Part 1 checks.
check = pd.read_csv("spate198_project_summary.csv", sep=";")
assert list(check.columns) == [
    "project_wocid", "commit_sha1", "author", "time", "commit message"
]
assert check["project_wocid"].nunique() == 10
assert len(check) == 29145
assert check[["commit_sha1", "author", "time"]].notna().all().all()
print(f"Checked {len(check):,} rows from {check['project_wocid'].nunique()} projects")

Checked 29,145 rows from 10 projects
